In [18]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 32
#define TM 8
#define TN 8

// ---- KERNEL CUDA ----
__global__ void sgemm_float4_only(int pad_n, const float *A, const float *B, float *C) {

    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * (BN / TN) + tx; // Da 0 a 255 (256 thread in totale)

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    for (int p = 0; p < (pad_n + BK - 1) / BK; ++p) {

        // Loading on Shared Memory through float4
        for (int i = 0; i < (BM * BK) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int a_row = load_idx / 4;
            int a_col_v = (load_idx % 4) * 4;

            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col_v;

            float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_a_row < pad_n && g_a_col < pad_n) {
                tmpA = *(const float4*)&A[g_a_row * pad_n + g_a_col];
            }
            *(float4*)&sA[a_row][a_col_v] = tmpA;
        }

        // Loading on Shared Memory through float4
        for (int i = 0; i < (BK * BN) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int b_row = load_idx / 32;
            int b_col_v = (load_idx % 32) * 4;

            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col_v;

            float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_b_row < pad_n && g_b_col < pad_n) {
                tmpB = *(const float4*)&B[g_b_row * pad_n + g_b_col];
            }
            *(float4*)&sB[b_row][b_col_v] = tmpB;
        }

        __syncthreads();

        // Computation on Shared Memory and Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // Write in Global Memory
    int row_start = by * BM + ty * TM;
    int col_start = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row_start + i;
            int c_col = col_start + j;
            if (c_row < pad_n && c_col < pad_n) {
                C[c_row * pad_n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) return 1;
    int n = atoi(argv[1]);

    // Arrounded for padding
    int pad_n = (n + BM - 1) / BM * BM;

    // Memory Allocation
    size_t pad_bytes = (size_t)pad_n * pad_n * sizeof(float);
    float *h_a = (float*)malloc(pad_bytes);
    float *h_b = (float*)malloc(pad_bytes);
    float *h_c = (float*)malloc(pad_bytes);

    // Padding loading
    for (int i = 0; i < pad_n; i++) {
        for (int j = 0; j < pad_n; j++) {
            if (i < n && j < n) {
                h_a[i * pad_n + j] = 2.0f;
                h_b[i * pad_n + j] = 3.0f;
            } else {
                h_a[i * pad_n + j] = 0.0f;
                h_b[i * pad_n + j] = 0.0f;
            }
            h_c[i * pad_n + j] = 0.0f;
        }
    }

    // CUDA Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, pad_bytes);
    cudaMalloc(&d_b, pad_bytes);
    cudaMalloc(&d_c, pad_bytes);

    // Data transferring
    cudaMemcpy(d_a, h_a, pad_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, pad_bytes, cudaMemcpyHostToDevice);

    // Setting the Grid and Block
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((pad_n + BN - 1) / BN, (pad_n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Timing
    cudaEventRecord(start);
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(pad_n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copying from GPU to CPU
    cudaMemcpy(h_c, d_c, pad_bytes, cudaMemcpyDeviceToHost);

    // Saving
    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * pad_n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_float4.cu', 'w') as f:
    f.write(cell_str)

In [23]:
!nvcc -arch=sm_75 -O3 cuda_float4.cu -o cuda_float4

In [24]:
!nvprof ./cuda_float4 20000

==4238== NVPROF is profiling process 4238, command: ./my_cuda_float4 20000
Execution Time: 3.5788 seconds
==4238== Profiling application: ./my_cuda_float4 20000
==4238== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   77.18%  3.57851s         1  3.57851s  3.57851s  3.57851s  sgemm_float4_only(int, float const *, float const *, float*)
                   14.93%  692.41ms         2  346.21ms  343.97ms  348.44ms  [CUDA memcpy HtoD]
                    7.89%  365.79ms         1  365.79ms  365.79ms  365.79ms  [CUDA memcpy DtoH]
      API calls:   74.23%  3.57852s         1  3.57852s  3.57852s  3.57852s  cudaEventSynchronize
                   21.97%  1.05910s         3  353.03ms  344.20ms  366.18ms  cudaMemcpy
                    3.63%  174.89ms         3  58.297ms  154.43us  174.58ms  cudaMalloc
                    0.12%  5.7244ms         3  1.9081ms  838.53us  2.7790ms  cudaFree
                    0.05%  2.4188ms     